In [1]:
import pandas as pd
import numpy as np
import requests
import time

In [2]:
import glob
import os

folder_path = r"ResaleFlatPrices"

csv_files = glob.glob(os.path.join(folder_path, "*.csv"))


df_list = []

for file in csv_files:
    filename = os.path.basename(file)

    if filename == "Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv":
        continue

    df = pd.read_csv(file)
    df_list.append(df)

df_hdb = pd.concat(df_list, ignore_index=True)

In [3]:
df_hdb

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0
...,...,...,...,...,...,...,...,...,...,...,...
685473,2012-02,YISHUN,5 ROOM,212,YISHUN ST 21,10 TO 12,121.0,Improved,1985,NaN,476888.0
685474,2012-02,YISHUN,5 ROOM,758,YISHUN ST 72,01 TO 03,122.0,Improved,1986,NaN,490000.0
685475,2012-02,YISHUN,5 ROOM,873,YISHUN ST 81,01 TO 03,122.0,Improved,1988,NaN,488000.0
685476,2012-02,YISHUN,EXECUTIVE,664,YISHUN AVE 4,07 TO 09,181.0,Apartment,1992,NaN,705000.0


In [ ]:
# Assuming your HDB dataframe is called df_hdb
# 1. Convert Month to Datetime and extract Year, quarter
df_hdb['month'] = pd.to_datetime(df_hdb['month'])
df_hdb['year'] = df_hdb['month'].dt.year
df_hdb['quarter'] = df_hdb['month'].dt.quarter

# 2. Extract numeric Remaining Lease (in years)
def clean_lease(lease_str):
    if isinstance(lease_str, str):
        parts = lease_str.split()
        years = int(parts[0])
        # Add months as a fraction of a year
        months = int(parts[2]) if len(parts) > 2 else 0
        return years + (months / 12)
    return lease_str

df_hdb['remaining_lease_years'] = df_hdb['remaining_lease'].apply(clean_lease)


# 3. Create Address column for Geocoding (Block + Street)
df_hdb['full_address'] = df_hdb['block'] + " " + df_hdb['street_name']

# 4. Log-transform Price (Common in Hedonic Models to normalize data)
df_hdb['log_resale_price'] = np.log(df_hdb['resale_price'])

df_hdb.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,year,quarter,remaining_lease_years,mid_storey,full_address,log_resale_price
0,2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0,2017,1,61.333333,11.0,406 ANG MO KIO AVE 10,12.354493
1,2017-01-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0,2017,1,60.583333,2.0,108 ANG MO KIO AVE 4,12.429216
2,2017-01-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0,2017,1,62.416667,2.0,602 ANG MO KIO AVE 5,12.476100
3,2017-01-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0,2017,1,62.083333,5.0,465 ANG MO KIO AVE 10,12.487485
4,2017-01-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0,2017,1,62.416667,2.0,601 ANG MO KIO AVE 5,12.487485


In [5]:

SEARCH_URL = "https://www.onemap.gov.sg/api/common/elastic/search"

def geocode_onemap(search_val, session=None, auth_token=None, retries=3, sleep=0.3):
    s = session or requests.Session()
    headers = {"Authorization": auth_token} if auth_token else {}
    params = {
        "searchVal": search_val,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }

    for i in range(retries):
        try:
            r = s.get(SEARCH_URL, params=params, headers=headers, timeout=15)
            r.raise_for_status()
            j = r.json()
            if int(j.get("found", 0)) > 0:
                best = j["results"][0]
                return float(best["LATITUDE"]), float(best["LONGITUDE"])
            return None, None
        except Exception:
            if i == retries - 1:
                return None, None
            time.sleep((i + 1) * sleep)



In [6]:
# cache unique addresses
addr_unique = df_hdb["full_address"].dropna().unique()
addr_map = {}
sess = requests.Session()

for a in addr_unique:
    addr_map[a] = geocode_onemap(a, session=sess)

df_hdb["lat"] = df_hdb["full_address"].map(lambda x: addr_map.get(x, (None, None))[0])
df_hdb["lon"] = df_hdb["full_address"].map(lambda x: addr_map.get(x, (None, None))[1])
df_hdb.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,year,quarter,remaining_lease_years,mid_storey,full_address,log_resale_price,lat,lon
0,2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0,2017,1,61.333333,11.0,406 ANG MO KIO AVE 10,12.354493,1.362005,103.853880
1,2017-01-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0,2017,1,60.583333,2.0,108 ANG MO KIO AVE 4,12.429216,1.370966,103.838202
2,2017-01-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0,2017,1,62.416667,2.0,602 ANG MO KIO AVE 5,12.476100,1.380709,103.835368
3,2017-01-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0,2017,1,62.083333,5.0,465 ANG MO KIO AVE 10,12.487485,1.366201,103.857201
4,2017-01-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0,2017,1,62.416667,2.0,601 ANG MO KIO AVE 5,12.487485,1.381041,103.835132


In [17]:
# check for na in any field
print(df_hdb.isna().sum())

month                         0
town                          0
flat_type                     0
block                         0
street_name                   0
storey_range                  0
floor_area_sqm                0
flat_model                    0
lease_commence_date           0
remaining_lease          421854
resale_price                  0
year                          0
quarter                       0
remaining_lease_years    421854
mid_storey                    0
full_address                  0
log_resale_price              0
lat                        3120
lon                        3120
dtype: int64


In [34]:
# Check to see if the % of null values for each of the town is significant
town_null_pct = (
    df_hdb
    .groupby("town")["lat"]
    .apply(lambda x: x.isna().mean() * 100)
)
town_null_pct.sort_values(ascending=False)

town
BUKIT MERAH        2.807376
CLEMENTI           2.581229
QUEENSTOWN         2.461783
CENTRAL AREA       1.774194
JURONG WEST        1.435113
JURONG EAST        0.944475
GEYLANG            0.879144
ANG MO KIO         0.398765
KALLANG/WHAMPOA    0.275551
WOODLANDS          0.182676
TOA PAYOH          0.098624
BEDOK              0.002400
CHOA CHU KANG      0.000000
BISHAN             0.000000
TAMPINES           0.000000
SERANGOON          0.000000
SENGKANG           0.000000
SEMBAWANG          0.000000
PUNGGOL            0.000000
BUKIT TIMAH        0.000000
PASIR RIS          0.000000
MARINE PARADE      0.000000
BUKIT BATOK        0.000000
HOUGANG            0.000000
BUKIT PANJANG      0.000000
YISHUN             0.000000
Name: lat, dtype: float64

In [19]:
# Removing those rows where we can't get lat and lon since they only consists of 3120 rows and it is roughly 2% of each of the affected towns which is a small subset of out data
df_hdb_clean = df_hdb.dropna(subset=['lat','lon'])

In [20]:
#Checking the null values in remaining lease
df_hdb_clean[df_hdb_clean['remaining_lease'].isna()]

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,year,quarter,remaining_lease_years,mid_storey,full_address,log_resale_price,lat,lon
226471,2012-03-01,ANG MO KIO,2 ROOM,172,ANG MO KIO AVE 4,06 TO 10,45.0,Improved,1986,NaN,250000.0,2012,1,NaN,8.0,172 ANG MO KIO AVE 4,12.429216,1.374694,103.836463
226472,2012-03-01,ANG MO KIO,2 ROOM,510,ANG MO KIO AVE 8,01 TO 05,44.0,Improved,1980,NaN,265000.0,2012,1,NaN,3.0,510 ANG MO KIO AVE 8,12.487485,1.373401,103.849073
226473,2012-03-01,ANG MO KIO,3 ROOM,610,ANG MO KIO AVE 4,06 TO 10,68.0,New Generation,1980,NaN,315000.0,2012,1,NaN,8.0,610 ANG MO KIO AVE 4,12.660328,1.379395,103.839157
226474,2012-03-01,ANG MO KIO,3 ROOM,474,ANG MO KIO AVE 10,01 TO 05,67.0,New Generation,1984,NaN,320000.0,2012,1,NaN,3.0,474 ANG MO KIO AVE 10,12.676076,1.362758,103.858015
226475,2012-03-01,ANG MO KIO,3 ROOM,604,ANG MO KIO AVE 5,06 TO 10,67.0,New Generation,1980,NaN,321000.0,2012,1,NaN,8.0,604 ANG MO KIO AVE 5,12.679196,1.379867,103.835977
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685473,2012-02-01,YISHUN,5 ROOM,212,YISHUN ST 21,10 TO 12,121.0,Improved,1985,NaN,476888.0,2012,1,NaN,11.0,212 YISHUN ST 21,13.075037,1.431783,103.837440
685474,2012-02-01,YISHUN,5 ROOM,758,YISHUN ST 72,01 TO 03,122.0,Improved,1986,NaN,490000.0,2012,1,NaN,2.0,758 YISHUN ST 72,13.102161,1.426030,103.834266
685475,2012-02-01,YISHUN,5 ROOM,873,YISHUN ST 81,01 TO 03,122.0,Improved,1988,NaN,488000.0,2012,1,NaN,2.0,873 YISHUN ST 81,13.098071,1.414695,103.836950
685476,2012-02-01,YISHUN,EXECUTIVE,664,YISHUN AVE 4,07 TO 09,181.0,Apartment,1992,NaN,705000.0,2012,1,NaN,8.0,664 YISHUN AVE 4,13.465953,1.420200,103.841075


In [21]:
# We assume that the lease commence on Jan 1st of each of the years

# convert month column to datetime
df_hdb_clean["month"] = pd.to_datetime(df_hdb_clean["month"], format="%Y-%m-%d")

# assume lease started on Jan 1 of lease_commence_date
lease_start = pd.to_datetime(df_hdb_clean["lease_commence_date"].astype(str) + "-01-01")

# compute remaining lease (years)
remaining_lease_years = 99 - ((df_hdb_clean["month"] - lease_start).dt.days / 365.25)

df_hdb_clean["remaining_lease_years"] = df_hdb_clean["remaining_lease_years"].fillna(remaining_lease_years)

/var/folders/2m/8llkplds47v3b1fv6f6vqv0m0000gn/T/ipykernel_1861/414824203.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hdb_clean["month"] = pd.to_datetime(df_hdb_clean["month"], format="%Y-%m-%d")
/var/folders/2m/8llkplds47v3b1fv6f6vqv0m0000gn/T/ipykernel_1861/414824203.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hdb_clean["remaining_lease_years"] = df_hdb_clean["remaining_lease_years"].fillna(remaining_lease_years)


In [22]:
print(df_hdb_clean.isna().sum())

month                         0
town                          0
flat_type                     0
block                         0
street_name                   0
storey_range                  0
floor_area_sqm                0
flat_model                    0
lease_commence_date           0
remaining_lease          418734
resale_price                  0
year                          0
quarter                       0
remaining_lease_years         0
mid_storey                    0
full_address                  0
log_resale_price              0
lat                           0
lon                           0
dtype: int64


In [26]:
# Remove unnecessary column
df_hdb_clean= df_hdb_clean.drop(columns=['remaining_lease'])

In [35]:
df_hdb_clean

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,year,quarter,remaining_lease_years,full_address,log_resale_price,lat,lon
0,2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,232000.0,2017,1,61.333333,406 ANG MO KIO AVE 10,12.354493,1.362005,103.853880
1,2017-01-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,250000.0,2017,1,60.583333,108 ANG MO KIO AVE 4,12.429216,1.370966,103.838202
2,2017-01-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,262000.0,2017,1,62.416667,602 ANG MO KIO AVE 5,12.476100,1.380709,103.835368
3,2017-01-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,265000.0,2017,1,62.083333,465 ANG MO KIO AVE 10,12.487485,1.366201,103.857201
4,2017-01-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,265000.0,2017,1,62.416667,601 ANG MO KIO AVE 5,12.487485,1.381041,103.835132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685473,2012-02-01,YISHUN,5 ROOM,212,YISHUN ST 21,10 TO 12,121.0,Improved,1985,476888.0,2012,1,71.917180,212 YISHUN ST 21,13.075037,1.431783,103.837440
685474,2012-02-01,YISHUN,5 ROOM,758,YISHUN ST 72,01 TO 03,122.0,Improved,1986,490000.0,2012,1,72.916496,758 YISHUN ST 72,13.102161,1.426030,103.834266
685475,2012-02-01,YISHUN,5 ROOM,873,YISHUN ST 81,01 TO 03,122.0,Improved,1988,488000.0,2012,1,74.915127,873 YISHUN ST 81,13.098071,1.414695,103.836950
685476,2012-02-01,YISHUN,EXECUTIVE,664,YISHUN AVE 4,07 TO 09,181.0,Apartment,1992,705000.0,2012,1,78.915127,664 YISHUN AVE 4,13.465953,1.420200,103.841075


In [ ]:
# Only keeping data from 2009 and onwards since we only have data from 2009 onwards
df_hdb_clean= df_hdb_clean[df_hdb_clean['month']>='2009-01-01']

In [40]:
df_hdb_clean

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,year,quarter,remaining_lease_years,full_address,log_resale_price,lat,lon
0,2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,232000.0,2017,1,61.333333,406 ANG MO KIO AVE 10,12.354493,1.362005,103.853880
1,2017-01-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,250000.0,2017,1,60.583333,108 ANG MO KIO AVE 4,12.429216,1.370966,103.838202
2,2017-01-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,262000.0,2017,1,62.416667,602 ANG MO KIO AVE 5,12.476100,1.380709,103.835368
3,2017-01-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,265000.0,2017,1,62.083333,465 ANG MO KIO AVE 10,12.487485,1.366201,103.857201
4,2017-01-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,265000.0,2017,1,62.416667,601 ANG MO KIO AVE 5,12.487485,1.381041,103.835132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685473,2012-02-01,YISHUN,5 ROOM,212,YISHUN ST 21,10 TO 12,121.0,Improved,1985,476888.0,2012,1,71.917180,212 YISHUN ST 21,13.075037,1.431783,103.837440
685474,2012-02-01,YISHUN,5 ROOM,758,YISHUN ST 72,01 TO 03,122.0,Improved,1986,490000.0,2012,1,72.916496,758 YISHUN ST 72,13.102161,1.426030,103.834266
685475,2012-02-01,YISHUN,5 ROOM,873,YISHUN ST 81,01 TO 03,122.0,Improved,1988,488000.0,2012,1,74.915127,873 YISHUN ST 81,13.098071,1.414695,103.836950
685476,2012-02-01,YISHUN,EXECUTIVE,664,YISHUN AVE 4,07 TO 09,181.0,Apartment,1992,705000.0,2012,1,78.915127,664 YISHUN AVE 4,13.465953,1.420200,103.841075


In [41]:
df_hdb_clean.to_csv('hdb_cleaned.csv', index=False)